In [ ]:
# Logging
import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings(action='ignore')

# Typing
from typing import Any, Generator, Iterable, Sequence, Optional, Union

# Stdlib
from ast import literal_eval

# I/O
import csv
import json
from pathlib import Path

# Numeric
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem

# Signac workflow control
import signac
from signac import Project
from signac.job import Job

from flow import FlowProject

## Formatting training data and breaking into jobs

In [ ]:
MONO_DATA_DIR = Path('monomer_data_formatted')
# N_TO_SAMPLE : Optional[int] = None
N_TO_SAMPLE : Optional[int] = 12
random : bool = True

mono_data_file_name = 'PolyID_master_data.csv'
mono_data_path = MONO_DATA_DIR / mono_data_file_name
assert(mono_data_path.exists())

READERS_BY_EXT = {
    '.xlsx' : pd.read_excel,
    '.csv'  : pd.read_csv,
}
df_reader_fn = READERS_BY_EXT[mono_data_path.suffix] # don't use get() here; WANT a KeyError if invalid
monomer_df = df_reader_fn(mono_data_path, index_col=0)
monomer_df.replace(np.nan, None, inplace=True) # convert NaN values to JSON-serializable NoneType

if N_TO_SAMPLE is not None:
    if random:
        monomer_df = monomer_df.sample(N_TO_SAMPLE)
    else:
        monomer_df = monomer_df.head(min(N_TO_SAMPLE, len(monomer_df)))
display(monomer_df)

## Generate jobs for each statepoint, injecting all other parameters not in the dataset

In [3]:
from typing import Any, Generator, Iterable, TypeAlias
StringMap : TypeAlias = dict[str, str]

import re
from itertools import product as cartesian_product


def cartesian_grid(param_options : dict[str, Iterable[Any]]) -> Generator[dict[str, Any], None, None]:
    '''
    Takes a dict keyed by parameter names whose values contain
    possible values for each respective parameter
    
    Exhaustively generates dicts (keyed by the same parameter names) containing every
    unique combination of those parameter values, with exactly one value for each key
    '''
    for param_point in cartesian_product(*param_options.values()):
        yield {
            param_name : param_value
                for param_name, param_value in zip(param_options.keys(), param_point)
        }

def parse_field_names_and_roles(dataframe : pd.DataFrame) -> tuple[StringMap, StringMap]:
    '''Extract the field (column) names and data role metadata
    Returns two dicts mapping from column names as-they-are to names and data roles, respectively'''

    HEADER_ROLE_RE = re.compile('(?P<field_name>.*?)<(?P<field_role>.*?)>') # role is delimited by chevrons

    colname_tag_free  : StringMap = {}
    colname_data_role : StringMap = {}
    for colname in dataframe.columns:
        matches = re.match(HEADER_ROLE_RE, colname)
        assert matches is not None

        match_fields : StringMap = matches.groupdict()
        colname_tag_free[colname] = match_fields['field_name']
        colname_data_role[colname] = match_fields['field_role']
        
    return colname_tag_free, colname_data_role

In [4]:
from collections import defaultdict

field_name, field_role = parse_field_names_and_roles(monomer_df)
gridspec = { # define other parameters to sweep here!
    'DOP' : (
        3,
        # 5,
    ),
    'n_atoms_max' : (
        10_000,
        # 20_000,
    ),
}

project_path = Path('polyid_test')
project = signac.init_project(project_path)

# populate data into statepoints
for i, row in monomer_df.iterrows():
    # divvy up values according to field role
    rowdata = defaultdict(dict)
    for field, value in row.items():
        rowdata[field_role[field]][field_name[field]] = value

    # generate job statepoints and metadata, inject shared state parameters as needed
    for shared_params in cartesian_grid(gridspec):
        statepoint = {**shared_params, **rowdata['statedata']} # make copies to avoi mutation of common data
        metadata   = {**rowdata['metadata']} # make copies to avoi mutation of common data

        job = project.open_job(statepoint=statepoint)
        job.document = metadata

# Skeletal Signac Project

#### Pre-defining blacklisted atoms and monomers

In [ ]:
from polymerist.polymers.monomers import specification
from polymerist.smileslib import substructures

BLACKLISTED_ATOM_QUERIES = {
    'silicon' : Chem.MolFromSmarts('[Si]'),
    'sulfur'  : Chem.MolFromSmarts('[S]'),
    'metal'   : substructures.SPECIAL_QUERY_MOLS['metal'],
    # 'halogen' : substructures.SPECIAL_QUERY_MOLS['halogen'],
}

BLACKLISTED_MONOMER_SMILES = [ # monomers which are, for one reason or another, disallowed
    'CC(C)(C)c1cc(c(Oc2ccc(cc2)N(c3ccc(N)cc3)c4ccc(N)cc4)c(c1)C(C)(C)C)C(C)(C)C',  # the extraordinary number of symmetries of this amine ("4-N-(4-aminophenyl)-4-N-[4-(2,4,6-tritert-butylphenoxy)phenyl]benzene-1,4-diamine")... 
    'CC(C)(C)c1cc(Oc2ccc(-c3ccc(N)cc3)cc2C(F)(F)F)c(C(C)(C)C)cc1Oc1ccc(-c2ccc(N)cc2)cc1C(F)(F)F' # ...mean it takes impractically long to isomorphism match during the Topology partition step
] # TODO: might try setting limit of <1000 automorphisms for automatic check (since this is the default limit for substructure matches)
BLACKLISTED_MONOMER_QUERIES = {}
for smiles in BLACKLISTED_MONOMER_SMILES:
    exp_spi = specification.expanded_SMILES(smiles, assign_map_nums=False)
    banned_mol = Chem.MolFromSmiles(exp_spi, sanitize=False)
    display(banned_mol)
    BLACKLISTED_MONOMER_QUERIES[smiles] = banned_mol

BLACKLISTED_MECHANISMS = [
    'imide',
    'vinyl'
]

In [28]:
from string import ascii_uppercase

from polymerist.genutils.textual.encoding import hash_as_alphanumeric

from polymerist.polymers.monomers import MonomerGroup
from polymerist.polymers.building import build_linear_polymer, mbmol_to_openmm_pdb

from polymerist.rdutils.reactions.reactions import AnnotatedReaction, BadNumberReactants
from polymerist.rdutils.reactions.reactors import PolymerizationReactor
from polymerist.rdutils import rdkdraw
rdkdraw.set_rdkdraw_size(300, 3/2)


class PolyIDBuild(FlowProject):
    pass


# 0) SIMPLE VALIDATION CHECKS ON REACTION DATA
def load_job_rdmol(job : Job) -> Chem.Mol:
    '''Helper method for loading an RDKit molecule from the SMILES in a job's statepoint'''
    reactant_mol = Chem.MolFromSmiles(job.sp.smiles_explicit, sanitize=False) # CRITICAL that sanitize=False to avoid stripping
    Chem.SanitizeMol(reactant_mol, sanitizeOps=specification.SANITIZE_AS_KEKULE) # single, unified mol containing individual reactant as disconnected components

    return reactant_mol

def load_job_rxn(job : Job) -> Chem.Mol:
    '''Helper method for loading an RDKit molecule from the SMILES in a job's statepoint'''
    rxn = AnnotatedReaction.from_smarts(
        job.sp.rxn_smarts.replace('#0', '*') # NOTE: this is a hack which should be sanitized in the previous rxn assembly step
    )
    rxn.Initialize()
    n_warn, n_err = rxn.Validate()
    # assert n_err == 0

    return rxn

@PolyIDBuild.label
def atoms_valid(job : Job) -> bool:
    '''Check no banned atom types are present'''
    # 2) check that none of the monomers are blacklisted
    reactant_mol = load_job_rdmol(job)
    return not any(
        substructures.matching_labels_from_substruct_dict(
            reactant_mol,
            BLACKLISTED_ATOM_QUERIES,
        )
    ) # if any illegal atoms are detected in the current monomer, return and exit

@PolyIDBuild.label
def monomers_allowed(job : Job) -> bool:
    '''Check no banned atom monomer fragments are present'''
    reactant_mol = load_job_rdmol(job)
    return not any(
        substructures.matching_labels_from_substruct_dict(
            reactant_mol,
            BLACKLISTED_MONOMER_QUERIES
        )
    ) # Exclude any monomers which are structurally disallowed

@PolyIDBuild.label
def mechanism_copied(job : Job) -> bool:
    return 'mechanism' in job.doc

@PolyIDBuild.label
def mechanism_allowed(job : Job) -> bool:
    '''
    Check that the rxn mechanism type is not explicitly blacklisted
    '''
    return job.doc.mechanism not in BLACKLISTED_MECHANISMS


# 1) TEST FOR RXN TEMAPLTE COMPLAINCE AND ENUMERATE CHEMICAL FRAGMENTS
polymerize = PolyIDBuild.make_group(name='polymerize')
FRAG_FILE_NAME = 'fragments.json'

@PolyIDBuild.label
def matches_rxn_template(job : Job) -> bool:
    return 'reactant_ordering' in job.doc

@polymerize
@PolyIDBuild.pre(atoms_valid)
@PolyIDBuild.pre(monomers_allowed)
@PolyIDBuild.pre(mechanism_copied)
@PolyIDBuild.pre(mechanism_allowed)
@PolyIDBuild.post(matches_rxn_template)
@PolyIDBuild.operation
def determine_reactant_order(job : Job) -> None:
    '''
    Check that SMILES monomers are compatible with the 
    reaction template for the mechanism they claim to follow
    '''
    # 1) check that monomers fit a reaction template
    reactant_mol = load_job_rdmol(job)
    reactants = Chem.GetMolFrags(reactant_mol, asMols=True)
    rxn = load_job_rxn(job)

    try:
        reactant_ordering = rxn.valid_reactant_ordering(reactants, as_mols=False)
        if reactant_ordering is None:
            return
        
        job.doc.reactant_ordering = reactant_ordering
    except BadNumberReactants:
        return 

ALLOWED_FUNCTIONALITIES : set[int] = {2}

@PolyIDBuild.label
def monomers_satisfy_functionality(job : Job) -> bool:
    '''
    Check that all monomers have allowed degrees of functionalization
    '''
    if not matches_rxn_template(job):
        return False

    rxn = load_job_rxn(job)
    reactant_smiles = job.sp.smiles_explicit.split('.')

    for i in job.doc.reactant_ordering:
        reactant_mol = Chem.MolFromSmiles(reactant_smiles[i], sanitize=False) # CRITICAL that sanitize=False to avoid stripping
        Chem.SanitizeMol(reactant_mol, sanitizeOps=specification.SANITIZE_AS_KEKULE) # single, unified mol containing individual reactant as disconnected components
        
        if substructures.num_substruct_queries_distinct(reactant_mol, rxn.GetReactantTemplate(i)) not in ALLOWED_FUNCTIONALITIES:
            return False
    else:
        return True
    
@polymerize
@PolyIDBuild.pre(matches_rxn_template)
@PolyIDBuild.pre(monomers_satisfy_functionality)
@PolyIDBuild.post.isfile(FRAG_FILE_NAME)
@PolyIDBuild.operation
def enum_fragments(job : Job) -> None:
    '''Enumerate all possible repeat unit fragment using cheminformatic reaction procedure'''
    rxn = load_job_rxn(job)
    reactor = PolymerizationReactor(rxn)

    reactant_mol = load_job_rdmol(job)
    reactants = Chem.GetMolFrags(reactant_mol, asMols=True)

    monogrp = MonomerGroup()
    for intermediates, frags in reactor.propagate(reactants):
        for assoc_group_name, rdfragment in zip(ascii_uppercase, frags):
            # generate spec-compliant SMARTS
            raw_smiles = Chem.MolToSmiles(rdfragment)
            exp_smiles = specification.expanded_SMILES(raw_smiles)
            spec_smarts = specification.compliant_mol_SMARTS(exp_smiles)

            # record to monomer group
            affix = 'TERM' if MonomerGroup.is_terminal(rdfragment) else 'MID'
            monogrp.monomers[f'{assoc_group_name}_{affix}'] = [spec_smarts]

    monogrp.to_file(job.fn(FRAG_FILE_NAME))


# 2) BUILD POLYMER STRUCTURE
oligomerize = PolyIDBuild.make_group(name='oligomerize')
OLIGOMER_PDB_NAME = 'oligomer.pdb'
OLIGOMER_SDF_NAME = 'oligomer.sdf'

@oligomerize
@PolyIDBuild.pre.copy_from(determine_reactant_order)
@PolyIDBuild.pre.copy_from(enum_fragments)
@PolyIDBuild.pre.isfile(FRAG_FILE_NAME)
@PolyIDBuild.post.isfile(OLIGOMER_PDB_NAME)
@PolyIDBuild.operation
def build_oligomer_pdb(job : Job) -> None:
    '''Generate coordinates and build oligomer PDB file using mBuild'''
    seq = 'BA' # hard-coded for now, plan to make more flexible in the future
    monogrp = MonomerGroup.from_file(job.fn(FRAG_FILE_NAME))
    polymer = build_linear_polymer(
        monomers=monogrp,
        DOP=2*(1 + (job.sp.DOP - 1)/len(seq)), # formula to convert target DOP (considering an AB pair as a repeat unit) to effective DOP in builder
        sequence=seq,
        energy_minimize=True, # TODO: add master config option for energy minimization at project level
    )
    mbmol_to_openmm_pdb(job.fn(OLIGOMER_PDB_NAME), polymer)

@PolyIDBuild.label
def matches_m2p_smiles(job : Job) -> bool:
    '''check whether the resulting polymer agrees with the SMILES output of M2P (if data is provided)'''
    pass

@oligomerize
@PolyIDBuild.pre.isfile(OLIGOMER_PDB_NAME)
@PolyIDBuild.post.isfile(OLIGOMER_SDF_NAME)
@PolyIDBuild.operation
def assign_chem_info(job : Job) -> None:
    '''Generate coordinates and build oligomer PDB file using mBuild'''
    pass

# @PolyIDBuild.pre.isfile(OLIGOMER_PDB_NAME)
# @PolyIDBuild.post(lambda : True) # TODO: fill this in with something more substantive!!
# @PolyIDBuild.operation
# def assign_partial_charges(job : Job) -> None:
#     '''Generate coordinates and build oligomer PDB file using mBuild'''
#     pass


# 3) PACK LATTICE
pack_lattice = PolyIDBuild.make_group(name='pack_lattice') 

In [29]:
project_path = Path('polyid_test')
project = PolyIDBuild.get_project(project_path)

In [ ]:
project.print_status(detailed='True')

#### Run Jobs

In [ ]:
project.run(
    names=['oligomerize']
)

In [ ]:
for job in project:
     if job.isfile(OLIGOMER_PDB_NAME):
          print(job)

In [16]:
job = project.open_job(id='b8e6a4735e555d650223d3196def8eb3')

## MD Engine file write

In [ ]:
from abc import ABC, abstractmethod
from openff.interchange import Interchange
from polymerist.genutils.decorators.classmod import register_subclasses


@register_subclasses(key_attr='ENGINE')
class MDEngineExporter(ABC):
    '''For simplifying the process of '''
    def __init_subclass__(cls, **kwargs) -> None:
        '''Enforce class-level definition of "Engine" name attr in subclasses'''
        super().__init_subclass__()
        if not hasattr(cls, 'ENGINE'):
            raise NotImplementedError('No class attr "ENGINE" set for subclass')
        
    def __init__(self, interchange : Interchange) -> None:
        super().__init__()
        self.interchange = interchange

    @property
    def inc(self) -> Interchange:
        '''Alias of "self.interchange" for convenience'''
        return self.interchange
    
    @abstractmethod
    def write_inputs(*args, **kwargs) -> None:
        pass

    
# Concrete classes
class LAMMPSMDExporter(MDEngineExporter):
    ENGINE = 'LAMMPS'

class OpenMMMDExporter(MDEngineExporter):
    ENGINE = 'OpenMM'